# Contact cutoffs at an interface — SKEMPI 2.0 at scale

**A companion notebook for [`foldenv`](https://pypi.org/project/foldenv/).** It answers one
question with 4,510 mutation records across 290 crystal complexes (323 carry single mutations;
33 of those contribute no scored record):

> When you count a residue's neighbours to describe its structural environment, should the
> cutoff be **Cα–Cα ≤ 8 Å** or **Cβ–Cβ ≤ 5 Å**?

`foldenv` already ships an answer — Cα-8 Å is decision **D1** in `foldenv/decisions.yaml`, chosen
on a monomer packing/burial argument. This notebook tests that choice on the one thing a monomer
cannot show you: **protein–protein interfaces**, where "is this residue in contact with something"
has an independent, experimentally curated ground truth.

It also measures the price of `foldenv`'s **monomer-only** view. The result is blunt:

| feature | AUROC (interface vs non-interface) | 95% CI |
|---|---|---|
| **cross-chain Cα-8 Å** | **0.690** | [0.678, 0.701] |
| cross-chain Cβ-5 Å | 0.600 | [0.591, 0.607] |
| monomer Cα-8 Å | 0.458 | [0.438, 0.479] |
| monomer Cβ-5 Å | 0.490 | [0.470, 0.511] |

Cα-8 Å wins by a margin whose confidence intervals do not overlap. And a monomer-only view of the
same residues predicts interface membership **at or below chance** (Cα-8 Å at 0.458, whose
interval excludes 0.5 — see §6) — the contacts that make a residue an
interface residue are exactly the ones a single-chain tool never sees.

Everything in the table above is **recomputed and asserted** below, not quoted. If a cell raises,
the notebook has failed to reproduce its own headline.

## Scope

Three things a reader should know before running it.

**1. The core measurement does not call `foldenv`.** The contact analysis is pure geometry — NumPy
and Biopython over crystal PDBs. (Two later sections do use the package: §8.2 borrows its MaxASA
table and residue-code map for the burial estimator, and §9 is a deliberate cross-check against its
contact routine. Neither is part of the measurement §5 reports.) It *validates a `foldenv` decision* (D1, the contact primary) without
*exercising `foldenv`'s code*, which makes it a slightly odd citizen in this repository. It earns
its place by being the evidence behind a shipped default, not by being a usage example.
[§9](#9) closes that gap deliberately: it re-runs the monomer half of the measurement through
`foldenv`'s own contact routine and shows the two agree residue-for-residue — and shows, in the
function signature, why that routine can never produce the cross-chain column.

**2. It measures a capability `foldenv` does not ship.** `foldenv` is monomer-only: one
AlphaFold chain in, one residue's environment out. The cross-chain arm below is a *complex*
computation. It informs a possible **complex-context extension**, and it tells a
user what their numbers cannot mean. It does not describe a feature you can call today.

**3. No data is redistributed here.** SKEMPI 2.0 and the crystal structures are downloaded at
runtime from their source ([§2](#2)). About 32 MB over the wire, ~121 MB unpacked.

*This analysis was first run as a set of standalone scripts. This notebook is now the record of
it: every headline it carries is recomputed and asserted here, and §7 adds four robustness
checks beyond them.*

## 1. Environment

Base `foldenv` is enough — `numpy` and `biopython` come with it. No PLM extra, no torch. The
`mkdssp` binary is **not** needed for the contact analysis (§3–§7) or the `foldenv` cross-check
(§9). §8.1 is the one section that needs it, and it says so; without `mkdssp` that section
skips and the rest of the notebook is unaffected.

In [1]:
# pip install foldenv        # brings numpy + biopython; no torch, no transformers
import platform, sys

import numpy as np
import Bio
import foldenv

print(f"python      {platform.python_version()} ({sys.platform})")
print(f"numpy       {np.__version__}")
print(f"biopython   {Bio.__version__}")
print(f"foldenv     {foldenv.__version__}")

python      3.12.13 (darwin)
numpy       2.5.3
biopython   1.88
foldenv     0.2.0


<a id="2"></a>
## 2. Data — downloaded, not shipped

Two files, both from the SKEMPI 2.0 distribution at the Barcelona Supercomputing Center:

| file | size | what it is |
|---|---|---|
| `skempi_v2.csv` | 1.6 MB | 7,085 mutation records: mutation, chain grouping, **structural class**, affinities |
| `SKEMPI2_PDBs.tgz` | 30 MB → 121 MB | 345 cleaned crystal complexes + their residue-numbering maps |

### Licence and attribution

**SKEMPI 2.0** supplies every mutation, every chain grouping, and every structure this notebook
runs on. Nothing here would exist without it.

> Jankauskaite, J., Jimenez-Garcia, B., Dapkunas, J., Fernandez-Recio, J. & Moal, I. H.
> "SKEMPI 2.0: an updated benchmark of changes in protein-protein binding energy, kinetics and
> thermodynamics upon mutation." *Bioinformatics* 35(3):462-469, 2019.
> doi:[10.1093/bioinformatics/bty635](https://doi.org/10.1093/bioinformatics/bty635)
>
> https://life.bsc.es/pid/skempi2

SKEMPI 2.0 is released under **[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)**, which
would permit redistribution with attribution. This notebook nonetheless **downloads** it rather
than vendoring it: the database
is BSC's to serve, it changes under them, and a copy in a git repository is a fork of someone
else's dataset that nobody asked for. **Protein Data Bank** structures reach you here via SKEMPI's
cleaned set; they are not redistributed by this repository either.

Nothing downloaded is modified. The derived numbers below are this notebook's own.

### The cleaned structure set, and why not RCSB

The structures come from SKEMPI's own `SKEMPI2_PDBs.tgz`, not from `files.rcsb.org`. That is a
correctness requirement, not a convenience: SKEMPI's per-mutation labels and its
`Mutation(s)_PDB` author numbering are defined **on this cleaned set**. Raw RCSB entries carry
waters, alternate locations, extra crystallographic chains and sometimes multiple models, all of
which change a neighbour count. Using them would silently produce different numbers against
labels that no longer describe the file.

### The download cell

It is **idempotent and resumable** — a complete file is skipped, a partial `.part` file resumes
with an HTTP `Range` request, and extraction only writes members that are missing. It also drops
the AppleDouble sidecars (`._1A22.pdb`) the archive carries from having been rolled on a Mac:
`tar` hides them, Python's `tarfile` does not, and left in place they would double every
directory count in this notebook. Re-running the cell after a complete run costs one `HEAD`
request in total: the structure branch is decided by counting `PDBs/` locally and asks the network
nothing, so only the CSV is checked. `SKEMPI_DATA_DIR` points it at an existing copy, which avoids
the download but still issues that one HEAD, so it needs a reachable network even when every file
is present.

In [2]:
import os, sys, tarfile, time, urllib.request
from pathlib import Path

DATA_DIR = Path(os.environ.get("SKEMPI_DATA_DIR", "skempi_data")).resolve()
PDB_DIR = DATA_DIR / "PDBs"
CSV_PATH = DATA_DIR / "skempi_v2.csv"
TGZ_PATH = DATA_DIR / "SKEMPI2_PDBs.tgz"

BASE = "https://life.bsc.es/pid/skempi2/database/download"
CSV_URL, TGZ_URL = f"{BASE}/skempi_v2.csv", f"{BASE}/SKEMPI2_PDBs.tgz"
N_COMPLEXES = 345          # .pdb files in SKEMPI2_PDBs.tgz


def _remote_meta(url: str) -> tuple[int | None, str | None]:
    """`(size, validator)` from a HEAD. The validator is the ETag, else Last-Modified."""
    req = urllib.request.Request(url, method="HEAD")
    with urllib.request.urlopen(req, timeout=60) as r:
        n = r.headers.get("Content-Length")
        tag = r.headers.get("ETag") or r.headers.get("Last-Modified")
    return (int(n) if n else None), tag


def fetch(url: str, dest: Path) -> Path:
    """Download `url` to `dest`, resuming a partial `.part` and skipping a complete file."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    total, validator = _remote_meta(url)
    if dest.exists() and (total is None or dest.stat().st_size == total):
        print(f"  {dest.name}: present ({dest.stat().st_size / 1e6:.1f} MB) — skipping")
        return dest

    part = dest.with_suffix(dest.suffix + ".part")
    have = part.stat().st_size if part.exists() else 0
    req = urllib.request.Request(url)
    if have:
        req.add_header("Range", f"bytes={have}-")
        # `If-Range` makes the resume conditional on the file not having changed: the server
        # returns 206 only if the validator still matches, and a plain 200 otherwise, which the
        # `resuming` check below turns into a clean restart. Without it, a re-rolled upstream
        # splices new bytes onto an old prefix and the size check cannot see it -- an equal-length
        # replacement passes. The dataset is served by a third party and does change.
        if validator:
            req.add_header("If-Range", validator)
        print(f"  {dest.name}: resuming at {have / 1e6:.1f} MB")

    t0, decile = time.time(), 0
    with urllib.request.urlopen(req, timeout=120) as r:
        resuming = r.status == 206
        mode = "ab" if (have and resuming) else "wb"
        done = have if (have and resuming) else 0
        with open(part, mode) as fh:
            while chunk := r.read(1 << 20):
                fh.write(chunk)
                done += len(chunk)
                if total and 10 * done // total > decile:      # progress, one line per 10%
                    decile = 10 * done // total
                    print(f"  {dest.name}: {done / 1e6:6.1f} / {total / 1e6:.1f} MB "
                          f"({100 * done / total:3.0f}%)", flush=True)
    print(f"  {dest.name}: {done / 1e6:.1f} MB in {time.time() - t0:.0f}s")
    if total and done != total:
        raise RuntimeError(f"{dest.name}: got {done} bytes, expected {total}")
    part.replace(dest)
    return dest


def structures(d: Path) -> list[Path]:
    """The real .pdb files in `d` — AppleDouble sidecars (`._1A22.pdb`) are not structures."""
    return sorted(p for p in d.glob("*.pdb") if not p.name.startswith("._"))


#: Ceiling on what one archive may expand to. The real one is ~121 MB unpacked; a 984:1
#: compression ratio is achievable, so an unbounded extract turns a 30 MB download into tens of
#: gigabytes on disk.
MAX_EXTRACT_BYTES = 512 * 1024 * 1024


def extract_missing(tgz: Path, into: Path) -> int:
    """Extract only members not already on disk. Returns the number written.

    Written defensively because the archive is fetched over the network: a tar can name a member
    outside the destination, or make it a symlink and write through it on a later pass.
    `isfile()` drops every symlink, hardlink, device and FIFO member, and flattening to
    `Path(...).name` leaves nothing for `..` or a leading `/` to act on.
    """
    into.mkdir(parents=True, exist_ok=True)
    written = 0
    total = 0
    with tarfile.open(tgz, "r:gz") as tf:
        kw = {"filter": "data"} if sys.version_info >= (3, 12) else {}
        for member in tf.getmembers():
            if not member.isfile():
                continue
            name = Path(member.name).name           # flatten "PDBs/1A22.pdb" -> "1A22.pdb"
            if not name or name in (".", ".."):
                continue                            # nothing usable after flattening
            if name.startswith("._") or name == ".DS_Store":
                continue    # the archive was rolled on a Mac: skip the AppleDouble sidecars,
                            # which GNU tar hides but Python's tarfile would happily write out
            if (into / name).exists():
                continue
            total += member.size
            if total > MAX_EXTRACT_BYTES:
                raise RuntimeError(
                    f"{tgz.name} expands past {MAX_EXTRACT_BYTES // (1024*1024)} MB; refusing"
                )
            member.name = name
            # `filter="data"` normalises permissions, but it is the default only from 3.12 and
            # this notebook supports older interpreters. Setting the mode unconditionally means
            # a hostile archive cannot land a setuid or world-writable file on any version.
            member.mode = 0o644
            tf.extract(member, into, **kw)
            written += 1
    return written


try:
    _shown = DATA_DIR.relative_to(Path.cwd())          # keep the printout short when it is
except ValueError:                                     # a subdirectory of the working dir
    _shown = DATA_DIR
print(f"data directory: {_shown}")
fetch(CSV_URL, CSV_PATH)

n_pdb = len(structures(PDB_DIR)) if PDB_DIR.exists() else 0
if n_pdb == N_COMPLEXES:
    print(f"  PDBs/: {n_pdb} structures present — skipping archive")
    # Presence is checked by name, not by size: a truncated file would pass here. It cannot pass
    # silently overall, because section 4 asserts the exact record and skip counts, which a short
    # structure would move -- but the failure would surface there rather than here.
else:
    fetch(TGZ_URL, TGZ_PATH)
    print(f"  extracting (have {n_pdb} of {N_COMPLEXES}) ...")
    print(f"  extracted {extract_missing(TGZ_PATH, PDB_DIR)} files")

pdb_files = structures(PDB_DIR)
assert len(pdb_files) == N_COMPLEXES, f"expected {N_COMPLEXES} structures, found {len(pdb_files)}"
mb = sum(p.stat().st_size for p in PDB_DIR.iterdir()) / 1e6
# The archive is optional once PDBs/ is populated, and the branch above skips fetching it
# when it is. Report its size only if it is actually here, so deleting it -- which the
# "if you keep it" below invites -- does not crash the first substantive cell on re-run.
kept = f", plus the {TGZ_PATH.stat().st_size / 1e6:.0f} MB archive if you keep it" \
    if TGZ_PATH.exists() else ""
print(f"\nready: {CSV_PATH.stat().st_size / 1e6:.1f} MB csv + {len(pdb_files)} structures "
      f"({mb:.0f} MB extracted{kept})")

data directory: skempi_data


  skempi_v2.csv: present (1.6 MB) — skipping
  PDBs/: 345 structures present — skipping archive

ready: 1.6 MB csv + 345 structures (121 MB extracted, plus the 30 MB archive if you keep it)


<a id="3"></a>
## 3. Ground truth — Levy's structural classes

SKEMPI's `iMutation_Location(s)` column gives every mutated residue a **Levy (2010) structural
class**, assigned from solvent accessibility in the monomer and in the complex. Five classes:

| class | | monomer | complex | interface? |
|---|---|---|---|---|
| `COR` | **core** | exposed | buried | **yes** |
| `RIM` | **rim** | exposed | still exposed, but loses area on binding | **yes** |
| `SUP` | **support** | buried | buried | **yes** |
| `INT` | **interior** | buried | buried, *unchanged* | **no** |
| `SUR` | **surface** | exposed | exposed, *unchanged* | **no** |

The definition is about **change on binding**: a residue is at the interface if the partner takes
solvent-accessible area away from it. `SUP` counts because it sits under the interface patch and
loses what little area it has; `INT` does not, because nothing about it changes when the partner
arrives.

> ### `INT` — an interior class, not an interface one
>
> This is the trap, and it is easy to fall into: "interior" sounds like "inside the complex."
> It is not. `INT` is **buried monomer interior** — the hydrophobic core of a single chain,
> typically far from any partner. It is easy to group
> `INT` with the interface classes; because `INT` residues have essentially **zero** cross-chain
> contacts, folding them into the positives poisons the positive class and pushes the cross-chain
> AUROC down. The corrected labelling is:
>
> ```python
> INTERFACE    = {"COR", "RIM", "SUP"}
> NONINTERFACE = {"INT", "SUR"}
> ```
>
> §5.2 is the standing check on this: `INT`'s mean cross-chain contact count must sit down with
> `SUR`, far below `COR`. If a future edit ever mislabels it again, that table shows it
> immediately.

### Selection

Only **single** mutations with a **single** unambiguous class are used: multi-point records carry
comma-separated mutations and classes, and a residue-level question has no answer for them. That
leaves 5,112 records from the 7,085 in the file.

In [3]:
import re
from collections import Counter, defaultdict

INTERFACE = {"COR", "RIM", "SUP"}        # Levy: core / rim / support
NONINTERFACE = {"INT", "SUR"}            # Levy: monomer interior / surface  <- NOT interface
CA_CUT, CB_CUT = 8.0, 5.0                # foldenv D1: primary Ca-8 A, alternative Cb-5 A

# "TI45G" -> wild-type T, chain I, residue 45 (+ optional insertion code), mutant G
_MUT = re.compile(r"^([A-Z])([A-Za-z0-9])(-?\d+[A-Za-z]?)([A-Z])$")


def parse_mutations(csv_path):
    """Single-mutation records with one clean Levy class, in file order.

    `#Pdb` is "<PDB>_<group1>_<group2>", the two sides of the binding equilibrium, e.g.
    "1CSE_E_I" = chain E against chain I. That grouping is what makes "cross-chain" well defined.
    """
    rows = []
    for line in csv_path.read_text().splitlines()[1:]:
        f = line.split(";")
        pdb_field, mut_pdb, _cleaned, loc = f[0], f[1], f[2], f[3]
        if "," in loc or not loc or "," in mut_pdb:      # singles only
            continue
        m = _MUT.match(mut_pdb)
        if not m:
            continue
        wt, chain, resnum_ic, _mt = m.groups()
        icode = resnum_ic[-1] if resnum_ic[-1].isalpha() else " "
        resnum = int(resnum_ic[:-1]) if icode != " " else int(resnum_ic)
        parts = pdb_field.split("_")
        if len(parts) != 3:                              # not a two-group complex
            continue
        rows.append({"pdb": parts[0], "g1": parts[1], "g2": parts[2], "chain": chain,
                     "resnum": resnum, "icode": icode, "wt": wt, "label": loc})
    return rows


MUTS = parse_mutations(CSV_PATH)
LABELLED = [r for r in MUTS if r["label"] in INTERFACE or r["label"] in NONINTERFACE]

counts = Counter(r["label"] for r in MUTS)
print(f"single-mutation records: {len(MUTS)}   with a clean Levy class: {len(LABELLED)}")
for cls in ("COR", "RIM", "SUP", "INT", "SUR"):
    side = "interface" if cls in INTERFACE else "non-interface"
    print(f"  {cls}  {counts[cls]:>5}   {side}")
print(f"distinct complexes carrying single mutations: {len({r['pdb'] for r in MUTS})} "
      f"(of {len(pdb_files)} structures downloaded)")

assert len(MUTS) == 5112, len(MUTS)
assert len(LABELLED) == 5112, "every single-mutation record should carry one of the five classes"

single-mutation records: 5112   with a clean Levy class: 5112
  COR   2237   interface
  RIM   1134   interface
  SUP    714   interface
  INT    398   non-interface
  SUR    629   non-interface
distinct complexes carrying single mutations: 323 (of 345 structures downloaded)


<a id="4"></a>
## 4. The framing fork — complex against monomer

An interface residue **only exists in a complex**. `foldenv` sees one chain. So there are two ways
to ask "which cutoff finds interface residues?", and they mean different things. This analysis
ran both, on purpose:

**A — the complex framing (the arm that literally answers the question).** Load the crystal
complex, and for each mutated residue count neighbours **on the partner chain(s)**, at Cα ≤ 8 Å
and at Cβ ≤ 5 Å. Score each cutoff by how well its count separates interface from non-interface
residues. This is a direct measurement of the cutoff question — and a computation `foldenv`
does not perform, since it never has a partner chain to count against.

**B — the monomer framing (the arm that measures the blindness).** Same residues, same cutoffs,
but count neighbours **on the residue's own chain only**, which is exactly the view a monomer-only
tool has. The expectation is a weak result. The value is in *how* weak, and in which direction.

Both arms come out of a single pass: for each residue, count neighbours and split them by whether
they sit on its own chain or the partner's.

### The monomer proxy: crystal chain, partner deleted

Arm B could have used AlphaFold monomer models — that is literally what `foldenv` consumes. It
uses the crystal chain with the partner deleted instead, for one reason: it isolates the variable.
An AF replication would mix "what the monomer view misses" with "what AlphaFold gets wrong", and
only the first is the question. The crystal chain is a *best-case* monomer — no prediction error
at all. Whatever it fails to see, an AF monomer fails to see at least as badly.

### Geometry

One representative atom per residue: **Cα** in `ca` mode, **Cβ** in `cb` mode with glycine falling
back to Cα (it has none) — the same rule as `foldenv`'s `contacts.glycine_cb_fallback: ca`.
Neighbours are counted as *distinct residues* within the cutoff, self excluded. A KD-tree
(Biopython's `NeighborSearch`) is built once per structure per mode and reused for every mutated
residue in it.

In [4]:
from Bio.PDB import NeighborSearch, PDBParser


def rep_atom(res, mode):
    """Ca for mode 'ca'; Cb for mode 'cb' with glycine -> Ca. None if the atom is absent."""
    want = "CA" if mode == "ca" else ("CA" if res.get_resname() == "GLY" else "CB")
    return res[want] if want in res else None


def build_ns(model, mode, amino_acids_only=False):
    """KD-tree over one representative atom per residue of the whole model (all chains)."""
    atoms = []
    for res in model.get_residues():
        if amino_acids_only and res.id[0].strip():       # skip HETATM records
            continue
        a = rep_atom(res, mode)
        if a is not None:
            atoms.append(a)
    return NeighborSearch(atoms)


def contacts_for(res, ns, own_chain, partner_chains, mode, cutoff):
    """(monomer, cross_chain) neighbour counts for `res`, or None if it has no rep atom."""
    a0 = rep_atom(res, mode)
    if a0 is None:
        return None
    mono = cross = 0
    for r in ns.search(a0.get_coord(), cutoff, level="R"):
        if r is res:
            continue
        ch = r.get_parent().id
        if ch == own_chain:
            mono += 1
        elif ch in partner_chains:
            cross += 1                # chains in neither binding group are ignored entirely
    return mono, cross


def score_dataset(muts, pdb_dir, amino_acids_only=False):
    """One pass over the complexes -> per-record contact counts + a skip tally.

    Records keep the CSV's order, which the bootstrap in §5 depends on for reproducibility.
    """
    by_pdb = defaultdict(list)
    for r in muts:
        by_pdb[r["pdb"]].append(r)                       # insertion order = file order

    parser = PDBParser(QUIET=True)
    recs, skipped = [], defaultdict(int)
    for pdb, rows in by_pdb.items():
        path = pdb_dir / f"{pdb}.pdb"
        if not path.exists():
            skipped["no_pdb"] += len(rows)
            continue
        try:
            model = parser.get_structure(pdb, str(path))[0]
        except Exception:
            skipped["parse_error"] += len(rows)
            continue
        chains = {c.id for c in model}
        ns_ca = build_ns(model, "ca", amino_acids_only)
        ns_cb = build_ns(model, "cb", amino_acids_only)
        for r in rows:
            if r["label"] not in INTERFACE and r["label"] not in NONINTERFACE:
                skipped["other_label"] += 1
                continue
            # the partner is the other side of SKEMPI's binding equilibrium
            g1, g2 = set(r["g1"]), set(r["g2"])
            if r["chain"] in g1:
                partners = g2 & chains
            elif r["chain"] in g2:
                partners = g1 & chains
            else:
                skipped["chain_not_in_groups"] += 1
                continue
            if r["chain"] not in chains or not partners:
                skipped["missing_chain"] += 1
                continue
            try:
                res = model[r["chain"]][(" ", r["resnum"], r["icode"])]
            except KeyError:
                skipped["residue_not_found"] += 1        # no density, or numbering mismatch
                continue
            ca = contacts_for(res, ns_ca, r["chain"], partners, "ca", CA_CUT)
            cb = contacts_for(res, ns_cb, r["chain"], partners, "cb", CB_CUT)
            if ca is None or cb is None:
                skipped["no_rep_atom"] += 1
                continue
            recs.append({"key": (r["pdb"], r["chain"], r["resnum"], r["icode"]),
                         "label": r["label"], "is_iface": int(r["label"] in INTERFACE),
                         "mono_ca": ca[0], "cross_ca": ca[1],
                         "mono_cb": cb[0], "cross_cb": cb[1]})
    return recs, dict(skipped)


t0 = time.time()
RECS, SKIPPED = score_dataset(LABELLED, PDB_DIR)
n_iface = sum(r["is_iface"] for r in RECS)

n_icode = sum(1 for r in LABELLED if r["icode"] != " ")
print(f"scored {len(RECS)} of {len(LABELLED)} records in {time.time() - t0:.1f}s")
print(f"  interface {n_iface}   non-interface {len(RECS) - n_iface}")
print(f"  skipped: {SKIPPED}")
print(f"  of those skips, {n_icode} are insertion-coded sites (e.g. residue 100A): SKEMPI's "
      "cleaned\n  structures carry no insertion code on any amino-acid residue (the only ones "
      "present sit on\n  glycan HETATMs), so those look-ups can never "
      "match. The rest\n  are residues with no density in the crystal.")
assert not [r for r in RECS if r["key"][3] != " "], "an insertion-coded residue was scored"

# --- reference values: what this analysis produced when it was first run, typed in as
# literals. Every value they are checked against is computed by the cells above.
assert len(RECS) == 4510, f"expected 4510 scored records, got {len(RECS)}"
assert n_iface == 3663, n_iface
assert len(RECS) - n_iface == 847, len(RECS) - n_iface
assert SKIPPED == {"residue_not_found": 602}, SKIPPED
print("\nOK — record counts match the reference run exactly")

scored 4510 of 5112 records in 8.6s
  interface 3663   non-interface 847
  skipped: {'residue_not_found': 602}
  of those skips, 35 are insertion-coded sites (e.g. residue 100A): SKEMPI's cleaned
  structures carry no insertion code on any amino-acid residue (the only ones present sit on
  glycan HETATMs), so those look-ups can never match. The rest
  are residues with no density in the crystal.

OK — record counts match the reference run exactly


<a id="5"></a>
## 5. Scoring — AUROC with bootstrap confidence intervals

**AUROC** is the right summary here. The features are *counts*, on incomparable scales — a
Cα-8 Å count runs to ~20, a Cβ-5 Å count rarely past 5 — so no fixed threshold could compare them
fairly. AUROC asks only a ranking question: **pick a random interface residue and a random
non-interface one; how often does the feature rank the interface one higher?** 0.5 is a coin flip,
1.0 is perfect, and below 0.5 means the feature points the wrong way.

It is computed from ranks (the Mann–Whitney U identity) with **ties shared**, which matters a lot
for integer counts: most of these residues have exactly 0 cross-chain neighbours (§6.1), and
letting their arbitrary sort order break ties would inflate the score.

The **95% CI** is a bootstrap: resample the 4,510 records with replacement 1,000 times, recompute
AUROC on each resample, take the 2.5th and 97.5th percentiles. Seeded (`default_rng(0)`), so the
interval is reproducible. Its job is to answer the only question that decides D1 — *is the Cα-8 Å
margin real, or could resampling produce it?*

In [5]:
def auroc(scores, labels):
    """P(random positive ranks above random negative), ties shared (Mann-Whitney U)."""
    pos, neg = labels == 1, labels == 0
    n_pos, n_neg = int(pos.sum()), int(neg.sum())
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    order = scores.argsort()
    ranks = np.empty(len(scores), float)
    ranks[order] = np.arange(1, len(scores) + 1)
    vals, inv, cnts = np.unique(scores, return_inverse=True, return_counts=True)
    sums = np.zeros(len(cnts))
    np.add.at(sums, inv, ranks)                  # average the ranks inside each tie group
    ranks = (sums / cnts)[inv]
    return (ranks[pos].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


def auroc_ci(scores, labels, n_boot=1000, seed=0):
    """(auroc, lo, hi) — percentile bootstrap over records."""
    rng = np.random.default_rng(seed)
    n = len(scores)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        a = auroc(scores[idx], labels[idx])
        if a == a:                               # skip a degenerate resample (all one class)
            boots.append(a)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return auroc(scores, labels), float(lo), float(hi)


FEATURES = {"cross_ca": "cross-chain  Ca-8 A", "cross_cb": "cross-chain  Cb-5 A",
            "mono_ca": "monomer      Ca-8 A", "mono_cb": "monomer      Cb-5 A"}


def score_features(recs, n_boot=1000):
    lab = np.array([r["is_iface"] for r in recs])
    return {k: auroc_ci(np.array([r[k] for r in recs], float), lab, n_boot=n_boot)
            for k in FEATURES}


t0 = time.time()
AUROCS = score_features(RECS)
print(f"{'feature':<22} {'AUROC':>7}   95% CI")
for k, name in FEATURES.items():
    a, lo, hi = AUROCS[k]
    print(f"{name:<22} {a:>7.3f}   [{lo:.3f}, {hi:.3f}]")
print(f"\n({time.time() - t0:.0f}s for 4 x 1000 bootstrap resamples)")

feature                  AUROC   95% CI
cross-chain  Ca-8 A      0.690   [0.678, 0.701]
cross-chain  Cb-5 A      0.600   [0.591, 0.607]
monomer      Ca-8 A      0.458   [0.438, 0.479]
monomer      Cb-5 A      0.490   [0.470, 0.511]

(3s for 4 x 1000 bootstrap resamples)


### 5.1 Reproduction check

The four values above are this analysis's headline. They are asserted against reference values
from its first run, to the full precision those were recorded at.

The point estimates are deterministic — no randomness enters them — and reproduced **exactly**
(to 1e-9) on two different NumPy/Biopython version pairs during this notebook's development. The
CI bounds depend on NumPy's PCG64 stream, which NumPy's versioning policy holds stable, so they
too came back bit-identical; they are asserted with a 2e-3 tolerance anyway, since a stream change
is a NumPy-side event and not evidence that this analysis has drifted.

In [6]:
# reference values from the first run of this analysis (see section 4)
GROUND_TRUTH = {
    "cross_ca": (0.690092636373628, 0.6784232411356288, 0.7013511611780316),
    "cross_cb": (0.5995498557482029, 0.591321432894661, 0.6066536157778644),
    "mono_ca": (0.45836616910997074, 0.43750688221015593, 0.47917803999728736),
    "mono_cb": (0.48987433284953946, 0.4696127405359859, 0.5108808636237063),
}

print(f"{'feature':<10} {'computed':>10} {'committed':>10} {'delta':>12}")
for k, (a_exp, lo_exp, hi_exp) in GROUND_TRUTH.items():
    a, lo, hi = AUROCS[k]
    print(f"{k:<10} {a:>10.6f} {a_exp:>10.6f} {a - a_exp:>12.2e}")
    assert abs(a - a_exp) < 1e-6, f"{k}: AUROC {a} != committed {a_exp}"
    assert abs(lo - lo_exp) < 2e-3 and abs(hi - hi_exp) < 2e-3, f"{k}: CI [{lo}, {hi}]"

# The decision itself: Ca-8 A beats Cb-5 A, and the intervals do not touch.
ca, cb = AUROCS["cross_ca"], AUROCS["cross_cb"]
assert ca[0] > cb[0] and ca[1] > cb[2], f"CIs overlap: Ca {ca} vs Cb {cb}"
print(f"\nOK — all four AUROCs and CIs reproduce")
print(f"OK — cross-chain Ca-8 A lower bound {ca[1]:.3f} > Cb-5 A upper bound {cb[2]:.3f} "
      f"(non-overlapping)")

feature      computed  committed        delta
cross_ca     0.690093   0.690093     0.00e+00
cross_cb     0.599550   0.599550     0.00e+00
mono_ca      0.458366   0.458366     0.00e+00
mono_cb      0.489874   0.489874     0.00e+00

OK — all four AUROCs and CIs reproduce
OK — cross-chain Ca-8 A lower bound 0.678 > Cb-5 A upper bound 0.607 (non-overlapping)


<a id="5.2"></a>
### 5.2 Per-class contacts — the Levy gradient, and the `INT` check

Averaging the Cα-8 Å counts within each Levy class shows what the AUROC compresses into one
number, and doubles as the standing check on the labelling from §3.

Read the cross-chain column top to bottom: `COR` (buried on binding, the middle of the patch) has
the most partner contacts; `RIM` and `SUP` have fewer; `INT` and `SUR` have almost none. Read the
monomer column and the ordering **falls apart** — `SUP` and `INT`, both buried in the monomer,
have the *highest* intra-chain packing, above `COR` and `RIM`, which are exposed until the partner
arrives. That inversion is the whole of finding B, visible before any AUROC is computed.

In [7]:
CLASSES = ["COR", "SUP", "INT", "RIM", "SUR"]
PER_CLASS_TRUTH = {           # committed: n, mean cross-chain Ca-8, mean monomer Ca-8
    "COR": (2033, 2.059517953762912, 9.01574028529267),
    "SUP": (631, 1.2297939778129952, 11.985736925515056),
    "INT": (320, 0.33125, 11.425),
    "RIM": (999, 1.2482482482482482, 8.393393393393394),
    "SUR": (527, 0.10056925996204934, 8.920303605313093),
}

print(f"{'class':<6} {'n':>5} {'cross-chain Ca-8':>28} {'monomer Ca-8':>26}")
per_class = {}
for c in CLASSES:
    rs = [r for r in RECS if r["label"] == c]
    mc = float(np.mean([r["cross_ca"] for r in rs]))
    mm = float(np.mean([r["mono_ca"] for r in rs]))
    per_class[c] = (len(rs), mc, mm)
    side = "iface" if c in INTERFACE else "  -  "
    bar_c = "#" * round(mc * 8)
    bar_m = "#" * round(mm * 2)
    print(f"{c} {side} {len(rs):>5}   {mc:4.2f} {bar_c:<20}   {mm:5.2f} {bar_m:<20}")

for c, (n_exp, mc_exp, mm_exp) in PER_CLASS_TRUTH.items():
    n, mc, mm = per_class[c]
    assert n == n_exp, f"{c}: n {n} != {n_exp}"
    assert abs(mc - mc_exp) < 1e-9 and abs(mm - mm_exp) < 1e-9, c

# The INT check: interior residues must sit with SUR on cross-chain contacts, not with COR.
assert per_class["INT"][1] < per_class["RIM"][1] / 2, "INT looks like an interface class -- labels?"
assert per_class["INT"][2] > per_class["COR"][2], "INT should out-pack COR in the monomer view"
print("\nOK — per-class means reproduce; INT sits with SUR on cross-chain contacts "
      "and above COR on monomer packing")

class      n             cross-chain Ca-8               monomer Ca-8
COR iface  2033   2.06 ################        9.02 ##################  
SUP iface   631   1.23 ##########             11.99 ########################
INT   -     320   0.33 ###                    11.43 #######################
RIM iface   999   1.25 ##########              8.39 #################   
SUR   -     527   0.10 #                       8.92 ##################  

OK — per-class means reproduce; INT sits with SUR on cross-chain contacts and above COR on monomer packing


<a id="6"></a>
## 6. Reading the numbers

**A — Cα-8 Å is the better interface cutoff, and the margin is real.** Cross-chain Cα-8 Å reaches
AUROC **0.690** [0.678, 0.701]; Cβ-5 Å reaches **0.600** [0.591, 0.607]. The intervals do not
touch. The mechanism is visible in the raw counts: Cβ ≤ 5 Å is so tight that a large share of
genuine interface residues have **zero** cross-chain Cβ neighbours, and a feature that is 0 for
much of the positive class cannot rank it. This agrees with the monomer-side packing verdict that
set `foldenv`'s D1 default in the first place — the same cutoff wins on both questions, for
related reasons.

That is a verdict on the shipped **pair**, not on the atom. The two arms differ in radius as well
as atom, and §7.5 separates them: at a matched 8 Å, Cβ beats Cα cross-chain (0.733 against 0.690,
paired Δ +0.043 with the interval excluding zero). So do **not** read this as settling what a
complex-context extension should use — on this evidence that question points the other way, and
§7.5 is where it is argued.

Note what the number is *not*: 0.690 is not "interface prediction accuracy". A bare neighbour
count with no burial term, no sequence, no evolutionary signal is a weak classifier in absolute
terms, and it is being asked to separate classes that overlap by construction (`RIM` residues are
barely buried on binding). The comparison between cutoffs is the result; the absolute value is
only the scale it lives on.

**B — the monomer view is blind to interfaces, and slightly worse than blind.** Monomer
intra-chain contacts predict interface membership at AUROC **0.458** (Cα-8 Å) and **0.490**
(Cβ-5 Å). Chance is 0.5. Cα-8 Å is *below* it, and the CI [0.438, 0.479] excludes 0.5, so this is
mild **anti**-prediction rather than noise: the per-class table says why. Interface core and rim
residues are surface-exposed in the isolated chain and therefore loosely packed, while the `INT`
negatives are buried in the hydrophobic core and tightly packed. Monomer packing does carry a
signal — it just points at burial, which at an interface runs *opposite* to the truth.

This is the quantified version of `foldenv`'s documented monomer-only limitation. A single-chain
tool cannot flag an interface residue from packing, because the contacts that define one are
exactly the contacts it never sees. And it is not merely uninformative: a user who reads a low
`contact_count` as "exposed, unimportant" will systematically misread interface cores.

### 6.1 The source of the modest absolute numbers

Both cutoffs were designed as *intra-chain* packing proxies, and 8 Å between Cα atoms is a
demanding requirement **across** an interface — two residues can be in intimate side-chain contact
with their Cα atoms 10 Å apart. So the honest picture of this feature is that it is zero for most
residues in both classes, and the AUROC is carried by the tail.

In [8]:
for name, grp in (("interface", [r for r in RECS if r["is_iface"] == 1]),
                  ("non-interface", [r for r in RECS if r["is_iface"] == 0])):
    z_ca = sum(r["cross_ca"] == 0 for r in grp) / len(grp)
    z_cb = sum(r["cross_cb"] == 0 for r in grp) / len(grp)
    print(f"{name:<14} n={len(grp):<5}  zero cross-chain Ca-8: {z_ca:5.1%}   "
          f"zero cross-chain Cb-5: {z_cb:5.1%}")
print()
for c in CLASSES:
    grp = [r for r in RECS if r["label"] == c]
    z_ca = sum(r["cross_ca"] == 0 for r in grp) / len(grp)
    z_cb = sum(r["cross_cb"] == 0 for r in grp) / len(grp)
    print(f"  {c}: zero at Ca-8 {z_ca:5.1%}   zero at Cb-5 {z_cb:5.1%}")

interface      n=3663   zero cross-chain Ca-8: 56.4%   zero cross-chain Cb-5: 78.6%
non-interface  n=847    zero cross-chain Ca-8: 93.7%   zero cross-chain Cb-5: 98.6%

  COR: zero at Ca-8 49.7%   zero at Cb-5 73.0%
  SUP: zero at Ca-8 70.0%   zero at Cb-5 91.1%
  INT: zero at Ca-8 91.2%   zero at Cb-5 97.8%
  RIM: zero at Ca-8 61.3%   zero at Cb-5 82.1%
  SUR: zero at Ca-8 95.3%   zero at Cb-5 99.1%


Two readings of this, both worth carrying away:

* **It explains the ranking.** At Cβ-5 Å, 79% of interface residues score zero, against 99% of
  non-interface ones — the classes are separated only by a 20-point difference in how often the
  feature fires at all. At Cα-8 Å the gap is 56% vs 94%. Loosening the cutoff recovers signal that
  the tighter one throws away, which is exactly why D1 keeps the looser cutoff.
* **It bounds the claim.** Roughly half of all `COR` residues — interface *cores* — have no
  cross-chain Cα neighbour within 8 Å. A contact count at this cutoff is a coarse interface
  detector even with the partner chain present. That is a limitation of the feature, not of the
  comparison between cutoffs, but a complex-context extension should treat 0.690 as a floor to
  improve on rather than a target it has met.

<a id="7"></a>
## 7. Robustness — four checks and a note on the counts

The headline reproduces. That is not the same as the headline being right, so here are five places
this analysis could have been fooled. Four of them are checks that run — §7.1, §7.2, §7.3 and
§7.5 — and §7.4 is a note on what the headline counts actually mean.

### 7.1 Calcium ions in the contact count

`build_ns` walks `model.get_residues()`, which in Biopython includes `HETATM` records. Most are
harmless — waters have no Cα or Cβ and drop out — but two groups do not:

* **modified amino acids** (`MSE`, `M3L`, `SEP`, `TPO`, `CGU`, ...) are stored as `HETATM` and have
  real Cα and Cβ atoms. Counting them as neighbours is *correct*: they are residues.
* **calcium ions** are stored as a residue named `CA` whose single atom is also named `CA`. In
  `ca` mode, `rep_atom` picks it up, and a bound Ca²⁺ is counted as a contacting residue. There
  are **106** of them across the 345 structures — 108 HETATM lines named `CA`, but two are altloc
pairs of a single ion that Biopython collapses into one residue, and it is residues that get
counted as contacts.

`foldenv` excludes both (`contacts._is_amino_acid` rejects any `HETATM`). Re-running with the same
exclusion measures whether it mattered.

In [9]:
t0 = time.time()
RECS_AA, SKIPPED_AA = score_dataset(LABELLED, PDB_DIR, amino_acids_only=True)
AUROCS_AA = score_features(RECS_AA)

assert len(RECS_AA) == len(RECS) and SKIPPED_AA == SKIPPED   # same residues, different neighbours
print(f"{'feature':<10} {'as published':>13} {'amino acids only':>18} {'delta':>10}")
for k in FEATURES:
    a, b = AUROCS[k][0], AUROCS_AA[k][0]
    print(f"{k:<10} {a:>13.4f} {b:>18.4f} {b - a:>10.4f}")
    assert abs(a - b) < 0.005, f"{k}: HETATM handling moves the result by {b - a:.4f}"
_worst = max(abs(AUROCS_AA[k][0] - AUROCS[k][0]) for k in FEATURES)
print(f"\n({time.time() - t0:.0f}s)  largest move {_worst:.4f} — the ion contacts are real "
      "but immaterial")

feature     as published   amino acids only      delta
cross_ca          0.6901             0.6900    -0.0001
cross_cb          0.5995             0.5995     0.0000
mono_ca           0.4584             0.4580    -0.0004
mono_cb           0.4899             0.4899     0.0000

(9s)  largest move 0.0004 — the ion contacts are real but immaterial


### 7.2 Pseudo-replication — records against distinct residues

SKEMPI is a database of **measurements**. The same mutation, measured by two labs or under two
conditions, is two rows — and different mutations *at the same position* (`TI45G`, `TI45A`,
`TI45W`) are three more. Every one of them contributes the same structural residue, with the same
contact counts and the same Levy class, as an independent point.

That inflates *n*, weights heavily-studied positions more, and — because the bootstrap resamples
rows — makes the confidence intervals narrower than the evidence warrants. The fix is to collapse
to unique `(pdb, chain, resnum, icode)` residues and rerun. Labels are consistent within a
residue, so the collapse is unambiguous.

In [10]:
seen, DEDUP = set(), []
for r in RECS:
    if r["key"] not in seen:
        seen.add(r["key"])
        DEDUP.append(r)

labels_per_residue = defaultdict(set)
for r in RECS:
    labels_per_residue[r["key"]].add(r["label"])
assert not [k for k, v in labels_per_residue.items() if len(v) > 1], "conflicting labels"

n_if = sum(r["is_iface"] for r in DEDUP)
print(f"records {len(RECS)}  ->  unique residues {len(DEDUP)} "
      f"(interface {n_if} / non-interface {len(DEDUP) - n_if})")
print(f"mean measurements per residue: {len(RECS) / len(DEDUP):.2f}\n")

AUROCS_DEDUP = score_features(DEDUP)
print(f"{'feature':<10} {'per record':>22} {'per residue':>22}")
for k in FEATURES:
    a, alo, ahi = AUROCS[k]
    b, blo, bhi = AUROCS_DEDUP[k]
    print(f"{k:<10} {a:>8.3f} [{alo:.3f}, {ahi:.3f}] {b:>8.3f} [{blo:.3f}, {bhi:.3f}]")

ca, cb = AUROCS_DEDUP["cross_ca"], AUROCS_DEDUP["cross_cb"]
assert ca[0] > cb[0] and ca[1] > cb[2], "the D1 verdict does not survive deduplication"
assert AUROCS_DEDUP["mono_ca"][2] < 0.52 and AUROCS_DEDUP["mono_cb"][2] < 0.52
print(f"\nOK — verdict A survives: Ca-8 A {ca[0]:.3f} [{ca[1]:.3f}, {ca[2]:.3f}] still beats "
      f"Cb-5 A {cb[0]:.3f} [{cb[1]:.3f}, {cb[2]:.3f}] with non-overlapping CIs")
print("OK — verdict B survives: both monomer features stay at or below chance")

records 4510  ->  unique residues 2609 (interface 1887 / non-interface 722)
mean measurements per residue: 1.73



feature                per record            per residue
cross_ca      0.690 [0.678, 0.701]    0.666 [0.652, 0.680]
cross_cb      0.600 [0.591, 0.607]    0.572 [0.562, 0.582]
mono_ca       0.458 [0.438, 0.479]    0.462 [0.437, 0.486]
mono_cb       0.490 [0.470, 0.511]    0.480 [0.457, 0.503]

OK — verdict A survives: Ca-8 A 0.666 [0.652, 0.680] still beats Cb-5 A 0.572 [0.562, 0.582] with non-overlapping CIs
OK — verdict B survives: both monomer features stay at or below chance


**Verdict: the conclusions hold, the headline number is optimistic.** Per residue, cross-chain
Cα-8 Å drops from 0.690 to ~0.666 and Cβ-5 Å from 0.600 to ~0.572; the gap between them barely
moves and the intervals still do not overlap. The monomer arm is unchanged, and still does not beat chance.

So D1 is safe, but the headline 0.690 ± 0.012 is a **measurement-weighted** figure, and its
interval is narrower than a residue-level analysis supports. Read the headline table as "AUROC
over SKEMPI single-mutation records", which is what it computes — not "over interface residues",
which is the easy misreading.

### 7.3 A corrupted structure — `1KBH`

The contact counts have a long tail. The median residue has 9 monomer Cα-8 Å neighbours and the
99th percentile has 17 — then a handful jump to 129, 202, 217. Every one of them is in **`1KBH`**.

`1KBH` is an NMR entry. SKEMPI's cleaned copy has had its 20 conformers **flattened into a single
chain each** — chain A holds 940 residues numbered 1–940 (20 × 47), chain B holds 1,180 (20 × 59),
with no `MODEL` records left to separate them. Twenty superimposed copies of the same protein sit
inside one 8 Å sphere, so every count from this file is inflated by roughly 20×, and its
cross-chain counts are contacts against 20 copies of the partner.

It is a known-bad entry elsewhere too: RDE-Network hard-codes a block list of exactly `{'1KBH'}`
and drops it before curation (Luo et al. 2023, cited under [Sources](#sources)). This analysis
keeps it, so the check below is worth running rather than assuming.

In [11]:
mono = np.array([r["mono_ca"] for r in RECS])
print(f"monomer Ca-8 counts: median {np.median(mono):.0f}  p99 {np.percentile(mono, 99):.0f}  "
      f"max {mono.max()}")
by_residue = {r["key"]: r for r in RECS}
worst = sorted(by_residue.values(), key=lambda r: -r["mono_ca"])[:5]
for r in worst:
    print(f"  {r['key'][0]} chain {r['key'][1]} {r['key'][2]:>4}  {r['label']}  "
          f"monomer {r['mono_ca']:>3}  cross-chain {r['cross_ca']:>3}")

CLEAN = [r for r in RECS if r["key"][0] != "1KBH"]
print(f"\ndropping 1KBH: {len(RECS)} -> {len(CLEAN)} records "
      f"({len(RECS) - len(CLEAN)} affected, {len({r['key'] for r in RECS if r['key'][0] == '1KBH'})} "
      "distinct residues)")

AUROCS_CLEAN = score_features(CLEAN)
print(f"\n{'feature':<10} {'as published':>13} {'1KBH dropped':>14} {'delta':>10}")
for k in FEATURES:
    a, b = AUROCS[k][0], AUROCS_CLEAN[k][0]
    print(f"{k:<10} {a:>13.4f} {b:>14.4f} {b - a:>10.4f}")
    assert abs(a - b) < 0.005, f"{k}: 1KBH moves the result by {b - a:.4f}"

ca, cb = AUROCS_CLEAN["cross_ca"], AUROCS_CLEAN["cross_cb"]
assert ca[1] > cb[2], "verdict A does not survive dropping 1KBH"
_worst1k = max(abs(AUROCS_CLEAN[k][0] - AUROCS[k][0]) for k in FEATURES)
print(f"\nOK — 9 records from a corrupted structure; largest move {_worst1k:.4f} and the "
      "verdict is unchanged.\n     Worth excluding on principle, not worth restating the "
      "results over.")

monomer Ca-8 counts: median 9  p99 17  max 217
  1KBH chain A   37  INT  monomer 217  cross-chain  21
  1KBH chain B   98  SUP  monomer 202  cross-chain  72
  1KBH chain A    3  SUP  monomer 129  cross-chain  20
  3BT1 chain U  120  INT  monomer  20  cross-chain   0
  3BT1 chain U  220  SUR  monomer  19  cross-chain   0

dropping 1KBH: 4510 -> 4501 records (9 affected, 3 distinct residues)



feature     as published   1KBH dropped      delta
cross_ca          0.6901         0.6911     0.0010
cross_cb          0.5995         0.6005     0.0009
mono_ca           0.4584         0.4584     0.0000
mono_cb           0.4899         0.4901     0.0002

OK — 9 records from a corrupted structure; largest move 0.0010 and the verdict is unchanged.
     Worth excluding on principle, not worth restating the results over.


### 7.4 What the headline counts mean

Two labelling points. Neither changes a number; both change what a number should be called.

1. **"Residues scored" counts records.** The headline "single-mutation residues scored: 4,510"
   describes 4,510 SKEMPI *records*, covering 2,609 distinct residues (§7.2). Wherever this
   notebook says "residues scored", read "records" — which is why §7.2 re-runs the AUROC one
   record per residue, and finds it holds.
2. **The negative class is `INT` + `SUR`, not `SUR` alone.** Of the 847 negatives, 320 are `INT`
   and 527 `SUR`, so an AUROC described as "interface vs surface" is really interface against
   both. §3's definition is the load-bearing one: `INTERFACE = {COR, RIM, SUP}`, everything else
   negative.

And one about the data rather than its description: `1KBH` (§7.3) should have been excluded.

### 7.5 Matched-radius control — atom against radius

§5 compares **Cα ≤ 8 Å** against **Cβ ≤ 5 Å**, which differ in the representative atom *and* in
the shell radius — so the 0.090 gap between them cannot be attributed to either one alone.
`foldenv`'s D1 is defined as that bundled pair, which makes the pair the right thing to validate
for a ship / don't-ship decision, but it leaves the two factors confounded: §6.1's explanation
("the tighter cutoff throws signal away") is a claim about the **radius**, and only the 2 × 2 can
test it.

Completing that square costs one extra neighbour search per record, against KD-trees that are
already built. These four arms carry no reference literals of their own — §5.1's reproduction
checks pin the two shipped arms and are untouched by this section.

The right statistic for "is A better than B on the same records" is a **paired** bootstrap on the
difference, not two independent intervals: the arms are measured on identical residues and are
strongly correlated, so overlap between separate CIs would understate the evidence.

In [12]:
ARMS = [("ca8", "ca", 8.0), ("cb5", "cb", 5.0), ("cb8", "cb", 8.0), ("ca5", "ca", 5.0)]


def score_arms(muts, pdb_dir):
    """One pass, four (atom, radius) arms. Reuses the helpers from section 4."""
    by_pdb = defaultdict(list)
    for r in muts:
        by_pdb[r["pdb"]].append(r)
    parser = PDBParser(QUIET=True)
    out = []
    for pdb, rows in by_pdb.items():
        path = pdb_dir / f"{pdb}.pdb"
        if not path.exists():
            continue
        try:
            model = parser.get_structure(pdb, str(path))[0]
        except Exception:
            continue
        chains = {c.id for c in model}
        ns = {m: build_ns(model, m) for m in ("ca", "cb")}   # built once, queried at both radii
        for r in rows:
            g1, g2 = set(r["g1"]), set(r["g2"])
            partners = g2 & chains if r["chain"] in g1 else (
                       g1 & chains if r["chain"] in g2 else None)
            if not partners or r["chain"] not in chains:
                continue
            try:
                res = model[r["chain"]][(" ", r["resnum"], r["icode"])]
            except KeyError:
                continue
            vals, ok = {}, True
            for name, mode, cut in ARMS:
                c = contacts_for(res, ns[mode], r["chain"], partners, mode, cut)
                if c is None:
                    ok = False
                    break
                vals[f"mono_{name}"], vals[f"cross_{name}"] = c
            if ok:
                out.append({"is_iface": int(r["label"] in INTERFACE), **vals})
    return out


t0 = time.time()
ARM_RECS = score_arms(LABELLED, PDB_DIR)
lab4 = np.array([r["is_iface"] for r in ARM_RECS])
assert len(ARM_RECS) == len(RECS), f"{len(ARM_RECS)} != {len(RECS)}"   # same residue set as above

print(f"{'arm':<26}{'AUROC':>8}   95% CI            ")
arm_auroc = {}
for pre, tag in (("cross", "cross-chain"), ("mono", "monomer    ")):
    for name, mode, cut in ARMS:
        a, lo, hi = auroc_ci(np.array([r[f"{pre}_{name}"] for r in ARM_RECS], float), lab4)
        arm_auroc[f"{pre}_{name}"] = (a, lo, hi)
        pub = "  <- published pair" if name in ("ca8", "cb5") else ""
        print(f"{tag + '  ' + mode.upper() + '-' + str(cut):<26}{a:>8.4f}   "
              f"[{lo:.3f}, {hi:.3f}]{pub}")
    print()

# the two arms section 5 reports must land in exactly the same place here
for pre, name, key in (("cross", "ca8", "cross_ca"), ("cross", "cb5", "cross_cb"),
                       ("mono", "ca8", "mono_ca"), ("mono", "cb5", "mono_cb")):
    assert abs(arm_auroc[f"{pre}_{name}"][0] - AUROCS[key][0]) < 1e-12, key
print("the two published arms reproduce section 5 exactly")

arm                          AUROC   95% CI            


cross-chain  CA-8.0         0.6901   [0.678, 0.701]  <- published pair


cross-chain  CB-5.0         0.5995   [0.591, 0.607]  <- published pair


cross-chain  CB-8.0         0.7326   [0.720, 0.744]


cross-chain  CA-5.0         0.5573   [0.552, 0.563]



monomer      CA-8.0         0.4584   [0.438, 0.479]  <- published pair


monomer      CB-5.0         0.4899   [0.470, 0.511]  <- published pair


monomer      CB-8.0         0.4503   [0.430, 0.471]


monomer      CA-5.0         0.4988   [0.480, 0.517]

the two published arms reproduce section 5 exactly


In [13]:
def paired_delta(a_key, b_key, recs, labels, n_boot=2000, seed=1):
    """Percentile bootstrap on AUROC(a) - AUROC(b), resampling records jointly."""
    A = np.array([r[a_key] for r in recs], float)
    B = np.array([r[b_key] for r in recs], float)
    rng = np.random.default_rng(seed)
    n, d = len(recs), []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        li = labels[idx]
        if li.sum() in (0, len(li)):
            continue
        d.append(auroc(A[idx], li) - auroc(B[idx], li))
    lo, hi = np.percentile(d, [2.5, 97.5])
    return auroc(A, labels) - auroc(B, labels), float(lo), float(hi)


print("paired bootstrap on the differences (cross-chain, 2000 resamples):\n")
CONTRASTS = [("cross_cb8", "cross_ca8", "atom, radius matched at 8 A   Cb - Ca"),
             ("cross_cb8", "cross_cb5", "radius, atom matched at Cb    8 A - 5 A"),
             ("cross_ca8", "cross_cb5", "the pair section 5 reports    Ca-8 - Cb-5")]
deltas = {}
for a, b, what in CONTRASTS:
    d, lo, hi = paired_delta(a, b, ARM_RECS, lab4)
    deltas[(a, b)] = (d, lo, hi)
    print(f"  {what:<40} {d:+.4f}  [{lo:+.4f}, {hi:+.4f}]  "
          f"{'excludes 0' if lo > 0 or hi < 0 else 'spans 0'}")

d_atom = deltas[("cross_cb8", "cross_ca8")]
d_rad = deltas[("cross_cb8", "cross_cb5")]
assert d_rad[1] > 0, "the radius effect should be positive and exclude zero"
assert d_atom[1] > 0, "at matched radius Cb should beat Ca, excluding zero"
assert d_rad[0] > d_atom[0], "the radius effect should dominate the atom effect"
print(f"\nasserted: radius effect ({d_rad[0]:+.4f}) exceeds the atom effect ({d_atom[0]:+.4f}), "
      "and\n          at a matched 8 A the atom effect favours Cb, not Ca")

paired bootstrap on the differences (cross-chain, 2000 resamples):



  atom, radius matched at 8 A   Cb - Ca    +0.0425  [+0.0345, +0.0505]  excludes 0


  radius, atom matched at Cb    8 A - 5 A  +0.1330  [+0.1225, +0.1433]  excludes 0


  the pair section 5 reports    Ca-8 - Cb-5 +0.0905  [+0.0821, +0.0991]  excludes 0

asserted: radius effect (+0.1330) exceeds the atom effect (+0.0425), and
          at a matched 8 A the atom effect favours Cb, not Ca


**The gap is mostly radius, and the atom term points the other way.** Decomposing the
+0.090 that §5 reports:

| | contrast | Δ AUROC |
|---|---|---|
| radius | Cβ-8 Å − Cβ-5 Å | **+0.133** |
| atom | Cβ-8 Å − Cα-8 Å | **+0.043**, favouring **Cβ** |
| published | Cα-8 Å − Cβ-5 Å | +0.090 |

The radius term is roughly three times the atom term, and at a matched 8 Å radius **Cβ-8 Å scores
0.733 against Cα-8 Å's 0.690** — a paired interval that excludes zero. So "Cα-8 Å is the better
interface cutoff" is true of the pair that was tested and misleading as a statement about the atom.

**This is not an argument to change D1, and it is not nothing either.** Three reasons it does not
license a change: this measures *cross-chain* contacts, while `foldenv` ships a monomer-only view —
and in the monomer arms above, Cβ-8 Å (0.450) is no better than Cα-8 Å (0.458), neither
beating chance; the
n counts records rather than distinct residues (§7.2); and a default has to satisfy the
*physical-sanity* gate on monomers, which this study does not test.

That gate is tested in the companion notebook, and it agrees with the decomposition here: both 8 Å
arms pass it and both 5 Å arms fail, for either atom. So across both studies the evidence says the
**radius** is what matters and the atom is a smaller, opposite-signed term. **Whether Cβ-8 Å would
be a better D1 default than Cα-8 Å is an open question neither study has answered** — it would need
a monomer-side benchmark with an outcome that is not itself a contact count.

<a id="8"></a>
## 8. The burial twin — `foldenv`'s headline signal

Contacts are one of the two structural signals `foldenv` returns. The other is **RSA**, relative
solvent accessibility, and it deserves the same test — partly as an independent check on finding
B, partly because RSA is the field most users actually read.

The burial version of the question is cleaner than the contact version, because it *is* Levy's
definition rather than a proxy for it:

$$\Delta\mathrm{rASA} = \mathrm{rASA}_{\text{monomer}} - \mathrm{rASA}_{\text{complex}}$$

— how much solvent-accessible area the partner takes away. A residue is at the interface exactly
when this is large. So two features, mirroring §4's two arms:

* **ΔrASA**, which needs both the isolated chain and the complex → the complex framing;
* **rASA in the monomer alone**, which is what `foldenv` returns today → the monomer framing.

Measured with **DSSP** in §8.1 below, ΔrASA reaches AUROC **0.750** [0.736, 0.765] against
monomer-only rASA at **0.549** [0.525, 0.573], over 3,669 scored records.

### 8.1 The DSSP implementation

The cell below is the DSSP implementation of that measurement: `mkdssp` on the complex, then
`mkdssp` again on each isolated chain written to a temporary PDB, then ΔrASA per residue.

**It runs here.** Executed against mkdssp 4.6.1, it returns ΔrASA at AUROC 0.750 [0.736, 0.765]
and monomer-only rASA at 0.549 [0.525, 0.573] over n = 3,669 — reproducing the reference values
to three decimals, bounds included. The cell still guards on `shutil.which("mkdssp")` and skips
with a message rather than pretending, so a reader without the binary gets the reference values
printed instead of asserted; but nothing here rests on that path.

Note its coverage: **979 records** are lost to complexes on which `mkdssp` fails outright (older
or multi-model PDB files) and another **464** to residues DSSP does not report, leaving 3,669 of
5,112 records scored — well short of the 4,510 the contact pass managed. Both counts are records
rather than residues or complexes: the code increments them by `len(rows)`, so the number of
failing *structures* is smaller. §8.2 runs a DSSP-free estimator instead, which has no such
failures, and the comparison follows it.

In [14]:
import shutil, tempfile
from pathlib import Path


def rsa_burial_dssp(muts, pdb_dir):
    """DeltarASA per record via mkdssp on the complex and on each isolated chain.

    DeltarASA from DSSP. Requires the `mkdssp` binary (DSSP v4); see section 8.1.
    """
    from Bio.PDB import DSSP, PDBIO, PDBParser, Select

    class _ChainSelect(Select):
        def __init__(self, chain_id):
            self.chain_id = chain_id

        def accept_chain(self, chain):
            return 1 if chain.id == self.chain_id else 0

    def _rasa(model, path):
        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            d = DSSP(model, str(path), dssp="mkdssp")
        out = {}
        for (chain_id, res_id), vals in d.property_dict.items():
            rasa = vals[3]                                  # relative ASA
            if rasa in ("NA", None):
                continue
            _het, resnum, icode = res_id
            try:
                out[(chain_id, int(resnum), icode)] = float(rasa)
            except (TypeError, ValueError):
                continue
        return out

    by_pdb = defaultdict(list)
    for r in muts:
        by_pdb[r["pdb"]].append(r)

    parser, io = PDBParser(QUIET=True), PDBIO()
    recs, skipped = [], defaultdict(int)
    with tempfile.TemporaryDirectory() as tmp:
        for pdb, rows in by_pdb.items():
            path = pdb_dir / f"{pdb}.pdb"
            if not path.exists():
                skipped["no_pdb"] += len(rows)
                continue
            try:
                struct = parser.get_structure(pdb, str(path))
                complex_rasa = _rasa(struct[0], path)
            except Exception:
                skipped["complex_dssp_error"] += len(rows)  # older / multi-model PDB files
                continue
            chain_rasa = {}
            for r in rows:
                ch = r["chain"]
                if ch in chain_rasa or ch not in {c.id for c in struct[0]}:
                    continue
                chain_pdb = Path(tmp) / f"{pdb}_{ch}.pdb"
                try:
                    io.set_structure(struct)
                    io.save(str(chain_pdb), _ChainSelect(ch))
                    chain_rasa[ch] = _rasa(parser.get_structure(f"{pdb}_{ch}", str(chain_pdb))[0],
                                           chain_pdb)
                except Exception:
                    chain_rasa[ch] = {}
                finally:
                    chain_pdb.unlink(missing_ok=True)
            for r in rows:
                key = (r["chain"], r["resnum"], r["icode"])
                rc, rm = complex_rasa.get(key), chain_rasa.get(r["chain"], {}).get(key)
                if rc is None or rm is None:
                    skipped["residue_no_dssp"] += 1
                    continue
                recs.append({"label": r["label"], "is_iface": int(r["label"] in INTERFACE),
                             "rasa_complex": rc, "rasa_monomer": rm, "delta_rasa": rm - rc})
    return recs, dict(skipped)


if shutil.which("mkdssp"):
    recs_dssp, skipped_dssp = rsa_burial_dssp(LABELLED, PDB_DIR)          # heavy: ~1000s of runs
    lab = np.array([r["is_iface"] for r in recs_dssp])
    for feat in ("delta_rasa", "rasa_monomer"):
        a, lo, hi = auroc_ci(np.array([r[feat] for r in recs_dssp], float), lab)
        print(f"{feat:<14} {a:.3f} [{lo:.3f}, {hi:.3f}]")
    print(f"n={len(recs_dssp)}  skipped={skipped_dssp}")
    print("compare with the reference run: delta 0.750 [0.736, 0.765], monomer 0.549 "
          "[0.525, 0.573], n=3669")
else:
    print("mkdssp not found — DSSP burial analysis SKIPPED; the reference values below stand in.")
    print("Install DSSP v4 to run it:  conda install -c conda-forge -c bioconda dssp")
    print("Section 8.2 below runs a DSSP-free substitute and does execute.")

delta_rasa     0.750 [0.736, 0.765]


rasa_monomer   0.549 [0.525, 0.573]
n=3669  skipped={'residue_no_dssp': 464, 'complex_dssp_error': 979}
compare with the reference run: delta 0.750 [0.736, 0.765], monomer 0.549 [0.525, 0.573], n=3669


### 8.2 A DSSP-free substitute

Solvent accessibility does not actually require DSSP. Biopython ships **Shrake–Rupley**, a
pure-Python numerical SASA, and `foldenv` exposes its own D3 normalisation table
(`foldenv.dssp.max_asa`, Tien et al. 2013 theoretical) — so rASA can be computed end to end with
no external binary:

```
rASA(residue) = SASA(residue) / MaxASA(residue type)
```

Isolated-chain SASA comes free: running Shrake–Rupley on a `Chain` entity considers only that
chain's atoms, which is exactly "the partner deleted". Complex first, chains second.

**This is a different estimator, not a reproduction.** DSSP and Shrake–Rupley disagree modestly on
absolute SASA, and the reference run's relative ASA used Biopython's `DSSP` normalisation rather
than Tien theoretical. The right expectation is agreement in *conclusion* and in the shape of the
per-class gradient, not agreement in the third decimal. What it buys is coverage: no `mkdssp`
failures, so all 4,510 records score instead of 3,669 (records in both cases).

Cost: ~6–8 minutes for the 323 complexes carrying single mutations, plus their mutated chains —
by far the slowest cell in the notebook, and the waters comparison after it costs the same again.
Set `RUN_BURIAL = False` at the top of the next cell to skip both.

In [15]:
RUN_BURIAL = True

from Bio.PDB.SASA import ShrakeRupley
from foldenv.constants import three2one
from foldenv.dssp import max_asa

MAX_ASA_TABLE = "tien2013_theoretical"          # foldenv D3 default


def rsa_burial_sasa(muts, pdb_dir, progress_every=60, strip_hetatm=True):
    """DeltarASA per record from Shrake-Rupley SASA + foldenv's MaxASA table. No mkdssp.

    `strip_hetatm=False` keeps crystallographic waters, ions and ligands in the SASA
    calculation; section 8.2's closing note measures what that costs.
    """
    by_pdb = defaultdict(list)
    for r in muts:
        by_pdb[r["pdb"]].append(r)

    parser, sr = PDBParser(QUIET=True), ShrakeRupley(n_points=100)
    recs, skipped, t0 = [], defaultdict(int), time.time()
    for i, (pdb, rows) in enumerate(by_pdb.items()):
        path = pdb_dir / f"{pdb}.pdb"
        if not path.exists():
            skipped["no_pdb"] += len(rows)
            continue
        model = parser.get_structure(pdb, str(path))[0]
        if strip_hetatm:                                              # protein atoms only:
            for ch in model:                                          # drop waters, ions,
                for rid in [r.id for r in ch if r.id[0].strip()]:     # ligands (DSSP does too)
                    ch.detach_child(rid)
        chains = {c.id for c in model}

        sr.compute(model, level="R")                                  # SASA in the complex
        in_complex = {(r.get_parent().id, r.id): r.sasa for r in model.get_residues()}

        in_monomer = {}
        for ch in {r["chain"] for r in rows} & chains:
            sr.compute(model[ch], level="R")                          # SASA of that chain alone
            in_monomer[ch] = {(ch, r.id): r.sasa for r in model[ch]}

        for r in rows:
            if r["chain"] not in chains:
                skipped["missing_chain"] += 1
                continue
            try:
                res = model[r["chain"]][(" ", r["resnum"], r["icode"])]
            except KeyError:
                skipped["residue_not_found"] += 1
                continue
            mx = max_asa(three2one.get(res.get_resname().strip().upper(), ""), MAX_ASA_TABLE)
            if not mx:
                skipped["no_max_asa"] += 1                            # non-standard residue
                continue
            key = (r["chain"], (" ", r["resnum"], r["icode"]))
            sc, sm = in_complex.get(key), in_monomer.get(r["chain"], {}).get(key)
            if sc is None or sm is None:
                skipped["no_sasa"] += 1
                continue
            rc, rm = sc / mx, sm / mx
            recs.append({"label": r["label"], "is_iface": int(r["label"] in INTERFACE),
                         "rasa_complex": rc, "rasa_monomer": rm, "delta_rasa": rm - rc})
        if progress_every and i % progress_every == 0:
            print(f"  {i:>3}/{len(by_pdb)} complexes  {time.time() - t0:>4.0f}s", flush=True)
    return recs, dict(skipped)


if RUN_BURIAL:
    t0 = time.time()
    RECS_RSA, SKIPPED_RSA = rsa_burial_sasa(LABELLED, PDB_DIR)
    lab_rsa = np.array([r["is_iface"] for r in RECS_RSA])
    print(f"\nscored {len(RECS_RSA)} records in {time.time() - t0:.0f}s   skipped={SKIPPED_RSA}")

    BURIAL = {f: auroc_ci(np.array([r[f] for r in RECS_RSA], float), lab_rsa)
              for f in ("delta_rasa", "rasa_monomer")}
    print(f"\n{'feature':<34} {'AUROC':>7}   95% CI        committed (DSSP)")
    for f, label, committed in (("delta_rasa", "DeltarASA (needs the complex)", "0.750 [0.736, 0.765]"),
                                ("rasa_monomer", "rASA, monomer only", "0.549 [0.525, 0.573]")):
        a, lo, hi = BURIAL[f]
        print(f"{label:<34} {a:>7.3f}   [{lo:.3f}, {hi:.3f}]   {committed}")

    print(f"\n{'class':<6} {'n':>5} {'rASA mono':>10} {'rASA cplx':>10} {'DeltarASA':>10}")
    deltas = {}
    for c in CLASSES:
        rs = [r for r in RECS_RSA if r["label"] == c]
        deltas[c] = float(np.mean([r["delta_rasa"] for r in rs]))
        print(f"{c:<6} {len(rs):>5} {np.mean([r['rasa_monomer'] for r in rs]):>10.2f} "
              f"{np.mean([r['rasa_complex'] for r in rs]):>10.2f} {deltas[c]:>10.2f}")

    d, mo = BURIAL["delta_rasa"], BURIAL["rasa_monomer"]
    assert d[0] > 0.70, f"DeltarASA should identify interfaces strongly, got {d[0]:.3f}"
    assert mo[0] < 0.60, f"monomer-only rASA should be weak, got {mo[0]:.3f}"
    assert d[1] > mo[2], "the two CIs should not overlap"
    order = [deltas[c] for c in ("COR", "RIM", "SUP", "INT", "SUR")]
    assert order == sorted(order, reverse=True), f"burial-on-binding gradient broken: {deltas}"
    print("\nOK — burial-on-binding separates interface from non-interface; the monomer view "
          "does not;\n     the per-class gradient COR > RIM > SUP > INT > SUR holds")

    0/323 complexes     0s


   60/323 complexes    25s


  120/323 complexes    58s


  180/323 complexes    90s


  240/323 complexes   144s


  300/323 complexes   201s



scored 4510 records in 377s   skipped={'residue_not_found': 602}



feature                              AUROC   95% CI        committed (DSSP)
DeltarASA (needs the complex)        0.740   [0.727, 0.754]   0.750 [0.736, 0.765]
rASA, monomer only                   0.554   [0.533, 0.574]   0.549 [0.525, 0.573]

class      n  rASA mono  rASA cplx  DeltarASA
COR     2033       0.37       0.14       0.23
SUP      631       0.23       0.13       0.11
INT      320       0.22       0.17       0.04
RIM      999       0.40       0.27       0.13
SUR      527       0.36       0.34       0.02

OK — burial-on-binding separates interface from non-interface; the monomer view does not;
     the per-class gradient COR > RIM > SUP > INT > SUR holds


In [16]:
# Section 8.2's closing note: what do crystallographic waters, ions and ligands cost?
# Same estimator, same records, stripping disabled. Roughly as slow as the run above.
if RUN_BURIAL:
    t0 = time.time()
    B_WET, _ = rsa_burial_sasa(LABELLED, PDB_DIR, strip_hetatm=False)
    lab_w = np.array([r["is_iface"] for r in B_WET])
    wet_delta = auroc_ci(np.array([r["delta_rasa"] for r in B_WET], float), lab_w)
    wet_mono = auroc_ci(np.array([r["rasa_monomer"] for r in B_WET], float), lab_w)

    print(f"{'':<16}{'stripped':>22}{'waters kept':>22}")
    for name, dry, wet in (("delta_rasa", BURIAL["delta_rasa"], wet_delta),
                           ("rasa_monomer", BURIAL["rasa_monomer"], wet_mono)):
        print(f"{name:<16}{dry[0]:>10.3f} [{dry[1]:.3f}, {dry[2]:.3f}]"
              f"{wet[0]:>10.3f} [{wet[1]:.3f}, {wet[2]:.3f}]")

    assert wet_delta[0] < BURIAL["delta_rasa"][0], \
        "keeping waters should weaken the burial signal, not strengthen it"
    print(f"\n({time.time() - t0:.0f}s)  keeping them costs "
          f"{BURIAL['delta_rasa'][0] - wet_delta[0]:.3f} AUROC — ordered water buries a "
          "residue's\n         surface without being its binding partner, so stripping is the "
          "defensible choice.")
else:
    print("RUN_BURIAL is False -- the waters comparison is skipped along with 8.2")

    0/323 complexes     0s


   60/323 complexes    28s


  120/323 complexes    63s


  180/323 complexes    97s


  240/323 complexes   153s


  300/323 complexes   211s


                              stripped           waters kept
delta_rasa           0.740 [0.727, 0.754]     0.731 [0.717, 0.745]
rasa_monomer         0.554 [0.533, 0.574]     0.528 [0.507, 0.548]

(390s)  keeping them costs 0.009 AUROC — ordered water buries a residue's
         surface without being its binding partner, so stripping is the defensible choice.


The DSSP-free run lands at ΔrASA **0.740** [0.727, 0.754] against monomer-only rASA **0.554**
[0.533, 0.574], on a 23% larger record set than the DSSP pass managed. Both intervals overlap the
reference DSSP values (0.750 [0.736, 0.765] and 0.549 [0.525, 0.573]), and the per-class gradient
has the same shape (`COR` 0.23 · `RIM` 0.13 · `SUP` 0.11 · `INT` 0.04 · `SUR` 0.02, against
DSSP's 0.28 · 0.16 · 0.11 · 0.05 · 0.02). Two independent estimators of solvent accessibility,
run on overlapping but different subsets, agree that burial-on-binding finds interfaces and that
monomer burial does not.

*One methodological note, and the cell above measures it rather than asserting it:* leaving
crystallographic waters, ions and ligands in the SASA calculation instead of stripping them pulls
ΔrASA down — ordered water buries a residue's surface without being its binding partner. Stripping
them, as DSSP does, is both the more defensible choice and the one that lands closer to the DSSP
run.

It also confirms finding B from a second signal. Contacts said monomer ≈ 0.49; burial says
monomer ≈ 0.55. Neither is usable. **The monomer-only limitation is a property of the input, not
of the feature you compute from it** — no amount of cleverness applied to one chain recovers
information that requires the second.

<a id="9"></a>
## 9. Closing the loop — the same measurement through `foldenv`'s own code

Everything above was NumPy and Biopython: the measurement genuinely does not need `foldenv`, and
imports nothing from it. That leaves an obvious question open — *is the geometry above actually
the geometry `foldenv` ships?* This section answers it by running the monomer arm a second time
through the package's own contact routine and comparing residue by residue.

Three things get checked:

1. **The defaults under test are the shipped defaults.** D1 is read out of the installed package,
   not retyped here.
2. **`foldenv`'s contact count equals this notebook's monomer count** on every comparable residue.
3. **The cross-chain column is not reachable through that API** — not "hard", not "unimplemented",
   but excluded by the shape of the function.

On (3), note where the public entry point starts: `foldenv.get_contacts(uniprot_id, position)`
takes a **UniProt accession**, fetches the AlphaFold **monomer** model for it, and counts within
one chain. There is no argument in which a partner could arrive. The lower-level
`foldenv.contacts.compute_contacts` — the function `get_contacts` delegates to, used here so the
comparison can run on the SKEMPI crystals rather than on AlphaFold models — takes a structure, but
also takes exactly one `chain_id`, and builds its KD-tree over that chain alone.

In [17]:
import inspect

from foldenv import config
from foldenv.contacts import build_contact_index, compute_contacts

cfg = config.load()
print("foldenv decisions (D1/D2), as installed:")
print(f"  contacts.primary            {cfg['contacts']['primary']!r}   <- the decision this "
      "notebook validates")
print(f"  contacts.ca_cutoff          {cfg['contacts']['ca_cutoff']} A")
print(f"  contacts.cb_cutoff          {cfg['contacts']['cb_cutoff']} A")
print(f"  contacts.glycine_cb_fallback {cfg['contacts']['glycine_cb_fallback']!r}")
print(f"  plddt.mask_below            {cfg['plddt']['mask_below']}")

assert cfg["contacts"]["primary"] == "ca", "D1 primary is no longer Ca"
assert cfg["contacts"]["ca_cutoff"] == CA_CUT and cfg["contacts"]["cb_cutoff"] == CB_CUT
assert cfg["contacts"]["glycine_cb_fallback"] == "ca"
print("\nOK — the shipped D1 default is Ca-8 A, which is what the 0.690-vs-0.600 comparison "
      "supports")

print(f"\ncompute_contacts{inspect.signature(compute_contacts)}")
print(f"build_contact_index{inspect.signature(build_contact_index)}")
print("\n-> one `chain_id`, no `partner_chains`: cross-chain counting is outside the type of "
      "this API,\n   and the public entry point above it takes a UniProt accession -> one "
      "AlphaFold monomer.")

foldenv decisions (D1/D2), as installed:
  contacts.primary            'ca'   <- the decision this notebook validates
  contacts.ca_cutoff          8.0 A
  contacts.cb_cutoff          5.0 A
  contacts.glycine_cb_fallback 'ca'
  plddt.mask_below            50.0

OK — the shipped D1 default is Ca-8 A, which is what the 0.690-vs-0.600 comparison supports

compute_contacts(structure, resnum: 'int', *, chain_id: 'str | None' = None, mode: 'str' = 'ca', ca_cutoff: 'float' = 8.0, cb_cutoff: 'float' = 5.0, n_nearest: 'int' = 5, plddt_mask_below: 'float' = 50.0, glycine_cb_fallback: 'str' = 'ca', index=None) -> 'ContactResult'
build_contact_index(structure, *, chain_id: 'str | None' = None, mode: 'str' = 'ca', plddt_mask_below: 'float' = 50.0, glycine_cb_fallback: 'str' = 'ca')

-> one `chain_id`, no `partner_chains`: cross-chain counting is outside the type of this API,
   and the public entry point above it takes a UniProt accession -> one AlphaFold monomer.


### 9.1 Residue-by-residue agreement

`compute_contacts` is pointed at each SKEMPI crystal chain that carries a mutation and asked for
the Cα-8 Å count of the mutated residue. The comparison is against the §7.1 amino-acids-only run,
since `foldenv` excludes `HETATM` records by design — comparing against the as-published counts
would just re-measure the calcium ions.

Two arguments are set away from their defaults, both for the same reason — **this is a crystal
structure, and `foldenv` is built for AlphaFold models**:

* `plddt_mask_below=0.0`. `foldenv`'s D2 default drops partner residues with pLDDT < 50, read from
  the B-factor column. In an AlphaFold model that column *is* pLDDT. In a crystal it is the
  **temperature factor**, where values below 50 are ordinary and mean well-ordered, not
  unreliable. Leaving the default on discards real neighbours: it changes the count for **79%** of
  these residues, by 6.7 neighbours on average. Anyone pointing `foldenv` at a crystal file needs
  to know this; the cell below measures it rather than asserting it.
* `n_nearest=0`, only to keep the output small.

Residues with an insertion code are skipped: `compute_contacts` finds a residue by number alone,
so `45` and `45A` are not distinguishable through it. In this dataset the guard never fires: as
§4 showed, SKEMPI's cleaned structures carry no insertion codes, so all 35 insertion-coded
mutation sites are already among the 602 unmatched residues.

In [18]:
mono_aa = {r["key"]: r["mono_ca"] for r in RECS_AA}          # amino-acids-only monomer counts

parser = PDBParser(QUIET=True)
by_pdb = defaultdict(list)
for r in RECS_AA:
    by_pdb[r["key"][0]].append(r["key"])

t0 = time.time()
n_match = n_diff = n_icode = 0
mask_delta, examples = [], []
for pdb, keys in by_pdb.items():
    struct = parser.get_structure(pdb, str(PDB_DIR / f"{pdb}.pdb"))
    for key in dict.fromkeys(keys):                          # unique residues, file order
        _pdb, chain, resnum, icode = key
        if icode != " ":
            n_icode += 1
            continue
        fe = compute_contacts(struct, resnum, chain_id=chain, mode=cfg["contacts"]["primary"],
                              ca_cutoff=cfg["contacts"]["ca_cutoff"], n_nearest=0,
                              plddt_mask_below=0.0,
                              glycine_cb_fallback=cfg["contacts"]["glycine_cb_fallback"])
        masked = compute_contacts(struct, resnum, chain_id=chain, n_nearest=0)   # D2 default on
        mask_delta.append(fe.contact_count - masked.contact_count)
        if fe.contact_count == mono_aa[key]:
            n_match += 1
        else:
            n_diff += 1
            if len(examples) < 5:
                examples.append((key, mono_aa[key], fe.contact_count))

md = np.array(mask_delta)
print(f"compared {n_match + n_diff} residues in {time.time() - t0:.0f}s "
      f"({n_icode} skipped for insertion codes)")
print(f"  foldenv contact_count == this notebook's monomer Ca-8 count:  {n_match}")
print(f"  disagreements:                                               {n_diff} {examples}")
print(f"\n  D2 pLDDT mask on crystal B-factors: changes {(md != 0).mean():.0%} of counts, "
      f"mean {md.mean():.1f} neighbours lost")

assert n_diff == 0, f"foldenv disagrees with this notebook on {n_diff} residues: {examples}"
print("\nOK — foldenv's shipped contact routine reproduces the monomer arm exactly.")
print("    It cannot reproduce the cross-chain arm at all: no argument admits a second chain.")

compared 2609 residues in 8s (0 skipped for insertion codes)
  foldenv contact_count == this notebook's monomer Ca-8 count:  2609
  disagreements:                                               0 []

  D2 pLDDT mask on crystal B-factors: changes 79% of counts, mean 6.7 neighbours lost

OK — foldenv's shipped contact routine reproduces the monomer arm exactly.
    It cannot reproduce the cross-chain arm at all: no argument admits a second chain.


### 9.2 One residue, both views

The abstract point, made concrete on a single interface core residue: what `foldenv` reports, and
what it cannot.

In [19]:
demo = next(r for r in RECS if r["label"] == "COR" and r["cross_ca"] >= 6
            and r["key"][3] == " " and r["mono_ca"] <= 8)
pdb, chain, resnum, _ic = demo["key"]
struct = parser.get_structure(pdb, str(PDB_DIR / f"{pdb}.pdb"))
fe = compute_contacts(struct, resnum, chain_id=chain, n_nearest=3, plddt_mask_below=0.0)

print(f"{pdb} chain {chain} residue {resnum} — SKEMPI class {demo['label']} (interface core)\n")
print(f"  foldenv (monomer view)   contact_count = {fe.contact_count:>2}   "
      f"atom_mode={fe.atom_mode}, cutoff={fe.cutoff} A")
for c in fe.nearest_contacts:
    print(f"      {c.aa}{c.resnum} at {c.distance:.1f} A   (same chain)")
print(f"\n  this notebook, complex   cross-chain   = {demo['cross_ca']:>2}   partner-chain "
      f"neighbours foldenv never sees")
print(f"\n  A user reading contact_count={fe.contact_count} sees a loosely packed, "
      "exposed-looking residue.")
print("  It is the middle of a binding interface.")

1B2U chain D residue 36 — SKEMPI class COR (interface core)

  foldenv (monomer view)   contact_count =  6   atom_mode=CA, cutoff=8.0 A
      A37 at 3.8 A   (same chain)
      L35 at 3.8 A   (same chain)
      W39 at 5.3 A   (same chain)

  this notebook, complex   cross-chain   =  6   partner-chain neighbours foldenv never sees

  A user reading contact_count=6 sees a loosely packed, exposed-looking residue.
  It is the middle of a binding interface.


<a id="10"></a>
## 10. Implications for a `foldenv` user

**Keep using Cα-8 Å, and do not read this as a result about Cα.** It is the default, and this is
the at-scale evidence for the *pair*: on 4,510 mutation records Cα-8 Å separates interface from
non-interface residues at 0.690 against 0.600 for Cβ-5 Å, with non-overlapping intervals, and the
ordering survives collapsing the data to unique residues (§7.2).

What §7.5 adds is that the gap is mostly the **radius**, not the atom. Cβ ≤ 5 Å is not "more
precise"; it is mostly zero, and a feature that is zero for 79% of its positive class cannot rank —
that is a statement about a 5 Å shell, and it holds for Cα at 5 Å too. Held at a matched 8 Å, the
atom term reverses: **Cβ-8 Å scores 0.733 to Cα-8 Å's 0.690**, paired interval excluding zero.

That is not a reason to change D1 — the arm that would justify a change is a *monomer* one, and
there neither atom beats chance — but it does mean **"Cβ-8 Å as the default" is an untested option
rather than a rejected one**, and it should not be described as settled by this study.

**Do not read `contact_count` or `rsa` as an interface signal.** This is the practical form of
finding B. For a residue on a binding face, `foldenv` returns numbers that look like *surface*,
because in the monomer that is exactly what it is:

| what you see for an interface core residue | what it means |
|---|---|
| low `contact_count` | loosely packed **in the monomer** — the partner is absent |
| high `rsa` | exposed **in the monomer** — the area is buried on binding |
| `secondary_structure` | unaffected; the fold is the fold |

Averaged over SKEMPI's interface residues, monomer contacts point *slightly the wrong way*
(AUROC 0.458) and monomer RSA is near chance (0.554). A pipeline that scores mutation sites by
burial will systematically rank interface cores as unimportant. If interfaces matter for your
question, the structural context of a monomer is not the input that answers it — and no better
statistic computed from that one chain will fix it.

**Where a monomer view is fine:** folded-core stability, packing, buried-versus-exposed for a
single chain, featurizing a mutation site where interface status is either irrelevant or supplied
from elsewhere. That is the tool's stated scope, and nothing here disturbs it.

### Scope and limits of this notebook

* **It tests a capability the package does not ship.** The cross-chain arm is a complex computation.
  It informs a possible **complex-context extension**; `foldenv` today is
  monomer-only and this notebook does not change that.
* **The monomer proxy is a crystal chain, not an AlphaFold model.** Deliberately: it isolates the
  monomer-view question from AlphaFold prediction error. An AF replication would add prediction
  noise on top, and could only make the monomer arm worse.
* **Ground truth is Levy's classes, not a physical interface definition.** They are derived from
  solvent accessibility, so the burial analysis in §8 is closer to circular than the contact
  analysis is; the contact result is the load-bearing one.
* **Records, not residues.** The published figures are measurement-weighted; §7.2 gives the
  residue-level numbers, which are slightly lower and hold the same conclusion.
* **§8.2's DSSP-free estimator is not a reproduction of §8.1.** It agrees with the DSSP
  measurement — 0.740 against 0.750 for ΔrASA — by a different route, which is corroboration
  rather than a repeat.

<a id="sources"></a>
### Sources

**SKEMPI 2.0** — every mutation, chain grouping and structure used above.

> Jankauskaite, J., Jimenez-Garcia, B., Dapkunas, J., Fernandez-Recio, J. & Moal, I. H.
> "SKEMPI 2.0: an updated benchmark of changes in protein-protein binding energy, kinetics and
> thermodynamics upon mutation." *Bioinformatics* 35(3):462-469, 2019.
> doi:[10.1093/bioinformatics/bty635](https://doi.org/10.1093/bioinformatics/bty635) ·
> https://life.bsc.es/pid/skempi2 · **CC BY 4.0**, downloaded not redistributed.

**Levy's structural classes** — the interface/non-interface ground truth.

> Levy, E. D. "A simple definition of structural regions in proteins and its use in analyzing
> interface evolution." *Journal of Molecular Biology* 403(4):660-670, 2010.
> doi:[10.1016/j.jmb.2010.09.028](https://doi.org/10.1016/j.jmb.2010.09.028)

**MaxASA normalisation** — `foldenv`'s D3 table, used in §8.2.

> Tien, M. Z., Meyer, A. G., Sydykova, D. K., Spielman, S. J. & Wilke, C. O. "Maximum allowed
> solvent accessibilites of residues in proteins." *PLoS ONE* 8(11):e80635, 2013.
> doi:[10.1371/journal.pone.0080635](https://doi.org/10.1371/journal.pone.0080635)

**Shrake–Rupley SASA** — via `Bio.PDB.SASA`, used in §8.2.

> Shrake, A. & Rupley, J. A. "Environment and exposure to solvent of protein atoms. Lysozyme and
> insulin." *Journal of Molecular Biology* 79(2):351-371, 1973.
> doi:[10.1016/0022-2836(73)90011-9](https://doi.org/10.1016/0022-2836(73)90011-9)

**Biopython** — structure parsing, KD-tree neighbour search, SASA.

> Cock, P. J. A. *et al.* "Biopython: freely available Python tools for computational molecular
> biology and bioinformatics." *Bioinformatics* 25(11):1422-1423, 2009.
> doi:[10.1093/bioinformatics/btp163](https://doi.org/10.1093/bioinformatics/btp163)

Structures are Protein Data Bank entries, obtained through SKEMPI's cleaned set and not
redistributed by this repository.

> Luo, S., Su, Y., Wu, Z., Su, C., Peng, J. & Ma, J. Rotamer density estimator is an unsupervised
> learner of the effect of mutations on protein-protein interaction. *ICLR 2023*. — RDE-Network;
> the source of the `{'1KBH'}` block list referenced in §7.3.
> [openreview.net/forum?id=_X9Yl1K2mD](https://openreview.net/forum?id=_X9Yl1K2mD)